In [ ]:
# Ability to automatically reload modules before executing code. This is useful for interactive development, as it allows you to modify your code and see the changes without restarting the kernel.
%load_ext autoreload
%autoreload 2

import os
import sys

# Add the parent directory to the Python path to allow importing modules from the src folder.
sys.path.append(os.path.abspath(os.path.join("..")))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# We do not import DataCleaner because EDA is performed on the raw data, before cleaning. However, we do import extract_age_in_days function from the preprocessing module, which is used to convert age to days.
from src.preprocessing import extract_age_in_days, TemporalFeaturesExtractor

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('data/raw_data/train.csv')

In [ ]:
counts = df['OutcomeType'].value_counts()
pct = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))

ax.set_title("Target Distribution (OutcomeType)", fontweight='bold')
ax.set_ylabel("Count")

offset_height = counts.max() * 0.02 

for bar, p in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset_height,
        f"{p:.1f}%",
        ha="center", 
        va="bottom", 
        fontsize=10
    )

plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


The 'OutcomeType' variable represents the target of the classification task. The distribution is not uniform, outcomes like 'Adoption' and 'Transfer' dominate,
while others are less represented. This indicates that the dataset was not obtained through stratified sampling and reflects real-world outcome frequencies at the Austin Animal Center.

In [ ]:
missing = df.isnull().sum()
print(missing)

In [ ]:
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values found in the dataset.")
else:
    
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot(kind="bar", ax=ax, color="salmon")
    
    ax.set_title("Missing Values by Column", fontweight='bold')
    ax.set_ylabel("Missing Values")
    
   
    offset_height = missing.max() * 0.02
    
    for i, v in enumerate(missing.values):
        ax.text(
            i, 
            v + offset_height, 
            str(v), 
            ha="center", 
            va="bottom", 
            fontsize=10
        )
        
    plt.tight_layout()
    plt.show()

**Handling Missing Values**

 'Name':
 We'll apply a univariate imputation strategy by replacing missing values (NaN) with the string 'Unknown'.
 This choice reflects that unnamed animals are likely strays or unowned, which can have behavioral or social implications.

'OutcomeSubtype':
 In principle, this variable could serve as an additional predictive target.
 However, due to the presence of nearly 50% missing entries, imputation is not appropriate
 and its inclusion would compromise data reliability.
 The feature is therefore excluded from further analysis to ensure consistent and interpretable results.

'SexuponOutcome':
 Only one value is missing in this feature.
 To balance functionality and statistical robustness, we replace the missing entry with the most frequent category (mode imputation).

'AgeuponOutcome':
 Only 18 values are missing in this feature.
 To balance functionality and statistical robustness, we replace the missing entries with the median.

In [ ]:
feature= 'AnimalType' 
counts = df[feature].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))

ax.set_title(f"Distribution by {feature}", fontweight='bold')
ax.set_ylabel("Count")

# 4. Etichette di testo con offset dinamico
offset_height = counts.max() * 0.02

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset_height,
        f"{int(bar.get_height())}", 
        ha="center", 
        va="bottom", 
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(df[feature], df['OutcomeType'], normalize="index")

fig, ax = plt.subplots(figsize=(10, 5))
ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")

ax.set_title(f"Outcome Distribution by {feature}", fontweight='bold')
ax.set_ylabel("Proportion")
ax.set_xlabel(feature)

plt.legend(title="Outcome", loc="upper right", bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.show()

In [ ]:
feature = 'Breed'
top_n = 40 

counts = df[feature].value_counts().head(top_n)
fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(counts.index, counts.values, color="salmon")

ax.set_title(f"Frequency of the first {top_n} categories for: {feature}", fontweight='bold')
ax.set_ylabel("Count")


total_unique = df[feature].nunique()
ax.text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique}", 
    transform=ax.transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)

plt.xticks(rotation=90) 
plt.tight_layout()
plt.show()

In [ ]:
feature = 'Color'
top_n = 40 

counts = df[feature].value_counts().head(top_n)
fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(counts.index, counts.values, color="salmon")

ax.set_title(f"Frequency of the first {top_n} categories for: {feature}", fontweight='bold')
ax.set_ylabel("Count")


total_unique = df[feature].nunique()
ax.text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique}", 
    transform=ax.transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)

plt.xticks(rotation=90) 
plt.tight_layout()
plt.show()

In [ ]:
df_eda= df.copy()
df_eda["age_in_days"] = extract_age_in_days(df_eda["AgeuponOutcome"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df_eda, x="age_in_days", ax=axes[0])
axes[0].set_title("Boxplot of Age in Days")
axes[0].set_xlabel("Age (days)")

sns.histplot(
    data=df_eda,
    x="age_in_days",
    ax=axes[1],
    bins=50,
)
axes[1].set_title("Distribution of Age in Days")
axes[1].set_xlabel("Age (days)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Percentiles of Age in Days:")
print(df_eda["age_in_days"].dropna().quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

**Age in Days (`age_in_days`) Distribution & Outlier Analysis**

* **Right-Skewed & Discretized:** The distribution is highly right-skewed, with most animals under 1.5 years of age (median < 500 days). The periodic spikes represent integer year boundaries (e.g., 365, 730 days) due to rounding in the original textual strings (e.g., "1 year", "2 years").
* **Biologically Plausible Outliers:** The boxplot flags values above ~2,600 days (~7 years) as statistical outliers. However, the maximum values (~7,300 days / 20 years) are biologically realistic for cats and dogs, indicating there are no impossible data entry errors.
* **Handling Outliers:** Since the age distribution is heavily right-skewed, we apply a logarithmic transformation ($\log(x + 1)$).  This "squashes" the extreme numeric scale to assist distance-based and linear models, while fully preserving data integrity and the ordinal relationship of the animals' life stages. 
  

In [ ]:
ime_extractor = TemporalFeaturesExtractor(datetime_col="DateTime")
df_eda = time_extractor.transform(df)


df_eda['Month'] = df_eda['DateTime'].dt.month

weekday_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df_eda['Weekday_Name'] = df_eda['Weekday'].map(weekday_map)
df_eda['Weekday_Name'] = pd.Categorical(
    df_eda['Weekday_Name'], 
    categories=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], 
    ordered=True
)

ct_month = pd.crosstab(df_eda['Month'], df_eda['OutcomeType'], normalize='index')
ct_weekday = pd.crosstab(df_eda['Weekday_Name'], df_eda['OutcomeType'], normalize='index')
ct_hour = pd.crosstab(df_eda['Hour'], df_eda['OutcomeType'], normalize='index')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ct_month.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', legend=False)
axes[0].set_title("Outcome by Month", fontsize=14)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis='x', rotation=90)


ct_weekday.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', legend=False)
axes[1].set_title("Outcome by Day of Week", fontsize=14)
axes[1].set_xlabel("Day")
axes[1].tick_params(axis='x', rotation=90)

ct_hour.plot(kind='bar', stacked=True, ax=axes[2], colormap='Set2', legend=False)
axes[2].set_title("Outcome by Hour", fontsize=14)
axes[2].set_xlabel("Hour")
axes[2].tick_params(axis='x', rotation=90)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.91, 0.85), title="OutcomeType", fontsize=11)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
df['clean_name'] = df['Name'].fillna("").astype(str).str.strip()
df['has_name'] = (df['clean_name'].str.len() > 0).astype(int)
df['name_length'] = df['clean_name'].str.len().astype(int)

ct_name = pd.crosstab(df['has_name'], df['OutcomeType'], normalize='index')

fig, ax = plt.subplots(figsize=(8, 5))
ct_name.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')

ax.set_title("Impact of Name Presence on Outcome", fontweight='bold')
ax.set_xlabel("Has a Name? (0 = No, 1 = Yes)")
ax.set_ylabel("Proportion")
plt.legend(title="Outcome", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
df_named = df[df['name_length'] > 0]

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=df_named, 
    x='OutcomeType', 
    y='name_length', 
    hue='OutcomeType', 
    palette='Set2', 
    legend=False, 
    ax=ax
)

ax.set_title("Distribution of Name Length by Outcome Type", fontweight='bold')
ax.set_xlabel("Outcome")
ax.set_ylabel("Name Length (Characters)")
ax.set_ylim(0, df_named['name_length'].quantile(0.99) + 2) 

plt.tight_layout()
plt.show()

The name lenght is not an informative feature and is therefore excluded from the feature engineering. 